# NLP & Sentiment Analysis (DecodeLabs Project 4)

**Objective:** classify product reviews as Positive or Negative using NLTK preprocessing, TF-IDF and Naive Bayes / Linear SVM.

> Not yet executed. Run top to bottom from the project root and write your observations under each output.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
!pip -q install nltk scikit-learn pandas matplotlib joblib
from src.preprocessing import ensure_nltk_resources, TextPreprocessor
ensure_nltk_resources()

## 2. Dataset: source, acquisition, inspection and cleaning
UCI *Sentiment Labelled Sentences* (Amazon subset). Labels: 0 = Negative, 1 = Positive.

In [ ]:
from src.data_loader import download_dataset, load_reviews, clean_reviews
download_dataset(['amazon'])
raw, malformed = load_reviews('amazon')
df, report = clean_reviews(raw, malformed)
report

## 3. Exploratory data analysis

In [ ]:
from src.eda import run_eda
from IPython.display import Image, display
stats = run_eda(df, ROOT/'reports'/'figures')
print(stats['class_counts']); print(stats['word_length_by_class'])
for f in ['class_distribution','review_length_distribution','review_length_by_sentiment']:
    display(Image(str(ROOT/'reports'/'figures'/f'{f}.png')))

**Observations (from the Colab run):** 1000 reviews loaded, 990 after cleaning, with 497 negative and 493 positive, so the classes are balanced and accuracy is a fair headline metric. Check the length plots for how review length differs by sentiment.

## 4. Text preprocessing
Negations are kept; lemmatisation is POS-guided.

In [ ]:
prep = TextPreprocessor()
for s in ['I am not happy with this phone!', "It didn't work and the batteries died", 'Great sound, works perfectly']:
    print(s, '->', prep.preprocess_text(s))

## 5. Train/test split and TF-IDF
The split is stratified with seed 42. TF-IDF is fitted inside the pipeline on training text only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
X_train, X_test, y_train, y_test = train_test_split(df['text'].tolist(), df['label'].to_numpy(), test_size=0.2, stratify=df['label'], random_state=42)
vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True)
M = vec.fit_transform(prep.transform(X_train))
print(type(M), M.shape, 'non-zero fraction:', M.nnz / (M.shape[0]*M.shape[1]))

## 6. Training, tuning and evaluation
`run_training` runs the whole workflow (same code as `python -m src.train`).

In [ ]:
from src.train import run_training
summary = run_training()
print(open(ROOT/'reports'/'results'/'RESULTS.md').read())

In [ ]:
for f in ['confusion_matrix_naive_bayes','confusion_matrix_linear_svm','model_comparison']:
    display(Image(str(ROOT/'reports'/'figures'/f'{f}.png')))

## 7. Error analysis and influential features

In [ ]:
import pandas as pd
err = pd.read_csv(ROOT/'reports'/'results'/'errors.csv')
display(err[~err['correct']])
display(pd.read_csv(ROOT/'reports'/'results'/'top_features.csv'))

**Error analysis (from the Colab run):** the SVM made 31 mistakes on 198 test reviews (15 false positives, 16 false negatives). Patterns in the misclassified reviews: negated problems in positive reviews ("did not have any problem"), mixed or conceding sentences, sarcasm and implied complaints, rare words and typos, and neutral list-style positives. Strongest features: `not` (negative), `great`, `good`, `best`, `love` (positive); bigrams such as `not work` and `not regret` behave as intended. Feature weights are associations in this dataset, not universal truths.

## 8. Predictions on new text
These examples are illustrative, not evaluation data.

In [ ]:
from src.predict import load_model, predict_review, DEMO_REVIEWS
model = load_model()
for r in DEMO_REVIEWS:
    print(predict_review(r, model=model))

## 9. Conclusions and limitations
**Results (Colab run, seed 42, 792 train / 198 test):** Linear SVM, selected by cross-validated F1 (0.8332 vs 0.8264 for Naive Bayes), scored accuracy 0.8434, F1 0.8426 and ROC-AUC 0.9085 on the held-out test set. Naive Bayes scored accuracy 0.8283, F1 0.8265, ROC-AUC 0.8960.

**Reading the gap:** one test review is about 0.5 points, so the SVM's lead is roughly 3 reviews and the models are close.

**Limitations:** small dataset, binary labels with no Neutral class, short single-sentence reviews, weakness on negated praise, sarcasm, mixed sentiment and rare words, some debatable labels, and English electronics-style reviews only.

**Next steps:** more data (`--subset all`), error-driven features, probability calibration for the SVM, and a Neutral class.